# Meta DS Interview Practice — Ads / Monetization

## Context
You're a Data Scientist on the Meta Ads team. You've been handed a table of daily ad campaign performance data. Your job is to analyze it and surface insights — just like you would in a real product review.

Work through each question below. Think out loud in comments as you go — interviewers want to follow your reasoning, not just see the final answer.

---

## The Dataset

| Column | Description |
|---|---|
| `campaign_id` | Unique ID for each ad campaign |
| `advertiser_id` | ID of the advertiser who ran the campaign |
| `date` | Date of the record (daily grain) |
| `impressions` | Number of times the ad was shown |
| `clicks` | Number of clicks on the ad |
| `spend` | Amount spent in USD |
| `conversions` | Number of purchases or signups attributed to the ad |

---

## Questions

### Q1 — Warm-up: Campaign Efficiency Metrics
Calculate the **click-through rate (CTR)** and **cost-per-click (CPC)** for each campaign across all dates. Return the **top 5 campaigns by CTR**.

- CTR = clicks / impressions
- CPC = spend / clicks

---

### Q2 — Aggregation & Grouping: Advertiser Performance
For each advertiser, calculate their **total spend**, **total conversions**, and **cost-per-conversion (CPA)**.
Flag any advertiser whose CPA is **more than 2 standard deviations above the mean** — these are underperformers.

- CPA = spend / conversions

---

### Q3 — Time Series: Trend Analysis
The ads team wants to understand weekly trends. Resample the data to a **weekly grain** and compute total impressions, clicks, spend, and conversions per week.
Then calculate **week-over-week (WoW) % change** in spend.

---

### Q4 — Diagnosis: Anomaly Detection
A PM comes to you and says: 'Something looks off — our conversion rate dropped last week.'

Using the data, **identify which specific campaigns or advertisers are driving the drop**. Walk through your approach in comments before writing the code.

---

### Bonus — Product Question (No Code)
After your analysis, the PM asks:

> 'We are seeing strong CTR but low conversion rates across several campaigns. What hypotheses would you form, and how would you investigate further?'

Write your answer as markdown in the cell below. Think about: audience targeting, landing page quality, ad creative, attribution, and what additional data you would want.

---

In [15]:
import pandas as pd
import numpy as np

np.random.seed(42)

# --- Config ---
n_campaigns = 20
n_advertisers = 6
date_range = pd.date_range(start='2024-10-01', end='2024-11-30', freq='D')

records = []

for campaign_id in range(1, n_campaigns + 1):
    advertiser_id = (campaign_id % n_advertisers) + 1
    base_impressions = np.random.randint(5000, 50000)
    base_ctr = np.random.uniform(0.01, 0.08)
    base_cvr = np.random.uniform(0.02, 0.12)
    base_cpc = np.random.uniform(0.30, 2.50)

    for date in date_range:
        # Inject a drop in conversions for 2 campaigns in the last week
        cvr_multiplier = 1.0
        if campaign_id in [3, 7] and date >= pd.Timestamp('2024-11-24'):
            cvr_multiplier = 0.2  # sharp conversion drop

        impressions = int(base_impressions * np.random.uniform(0.85, 1.15))
        clicks = int(impressions * base_ctr * np.random.uniform(0.9, 1.1))
        clicks = max(clicks, 1)
        spend = round(clicks * base_cpc * np.random.uniform(0.9, 1.1), 2)
        conversions = int(clicks * base_cvr * cvr_multiplier * np.random.uniform(0.8, 1.2))

        records.append({
            'campaign_id': f'C{campaign_id:03d}',
            'advertiser_id': f'A{advertiser_id:02d}',
            'date': date,
            'impressions': impressions,
            'clicks': clicks,
            'spend': spend,
            'conversions': conversions
        })

df = pd.DataFrame(records)
df['date'] = pd.to_datetime(df['date'])

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Campaigns: {df["campaign_id"].nunique()}, Advertisers: {df["advertiser_id"].nunique()}')
df.head(10)


Dataset shape: (1220, 7)
Date range: 2024-10-01 to 2024-11-30
Campaigns: 20, Advertisers: 6


,campaign_id,advertiser_id,date,impressions,clicks,spend,conversions
0,C001,A02,2024-10-01,18649,1329,1959.12,142
1,C001,A02,2024-10-02,21425,1708,2497.10,189
2,C001,A02,2024-10-03,22868,1649,2496.83,134
3,C001,A02,2024-10-04,19573,1505,2400.53,128
4,C001,A02,2024-10-05,21492,1526,2365.03,134
5,C001,A02,2024-10-06,20520,1660,2523.07,155
6,C001,A02,2024-10-07,21371,1487,2456.27,120
7,C001,A02,2024-10-08,18081,1508,2665.60,157
8,C001,A02,2024-10-09,19576,1377,2308.72,125
9,C001,A02,2024-10-10,18437,1409,2066.25,152


---
## Your Work Starts Here
### Q1 — Warm-up: Campaign Efficiency Metrics
Calculate the **click-through rate (CTR)** and **cost-per-click (CPC)** for each campaign across all dates. Return the **top 5 campaigns by CTR**.

- CTR = clicks / impressions
- CPC = spend / clicks

In [16]:
# calculate the ctr and cpc
df["ctr"] = df["clicks"]/df["impressions"]
df["cpc"] = df["spend"]/df["clicks"]

# take average for each campaign id
campaign_df = df.groupby("campaign_id")[["ctr", "cpc"]].mean()
campaign_df.sort_values("ctr", ascending=False)

# answer:
# C005, C001, C002, C020, C008

,ctr,cpc
campaign_id,,
C005,0.078875,0.527497
C001,0.075916,1.621379
C002,0.070430,1.136948
C020,0.069008,0.608918
C008,0.066023,1.735293
C012,0.059820,1.626554
C007,0.048777,1.509037
C014,0.048328,1.217628
C019,0.046983,0.594394


### Q2 — Aggregation & Grouping: Advertiser Performance
For each advertiser, calculate their **total spend**, **total conversions**, and **cost-per-conversion (CPA)**.
Flag any advertiser whose CPA is **more than 2 standard deviations above the mean** — these are underperformers.

- CPA = spend / conversions

In [17]:
import numpy as np

# get the total
q2_df = df.groupby("advertiser_id")[["spend", "conversions"]].sum().reset_index()

# calculate cpa
q2_df["cpa"] = q2_df["spend"]/q2_df["conversions"]

# calculate mean and std
m = np.mean(q2_df["cpa"])
sd = np.std(q2_df["cpa"])

threshold = m + 2*sd
q2_df["flag"] = q2_df["cpa"] > threshold
q2_df.head()

# answer:
# there are no underperformers

,advertiser_id,spend,conversions,cpa,flag
0,A01,316332.38,17758,17.813514,False
1,A02,442027.54,23341,18.937815,False
2,A03,237887.65,12471,19.075267,False
3,A04,102663.93,8058,12.740622,False
4,A05,120572.88,11631,10.366510,False


### Q3 — Time Series: Trend Analysis
The ads team wants to understand weekly trends. Resample the data to a **weekly grain** and compute total impressions, clicks, spend, and conversions per week.
Then calculate **week-over-week (WoW) % change** in spend.

In [21]:
df.head()

,campaign_id,advertiser_id,date,impressions,clicks,spend,conversions,ctr,cpc
0,C001,A02,2024-10-01,18649,1329,1959.12,142,0.071264,1.474131
1,C001,A02,2024-10-02,21425,1708,2497.10,189,0.079720,1.462002
2,C001,A02,2024-10-03,22868,1649,2496.83,134,0.072109,1.514148
3,C001,A02,2024-10-04,19573,1505,2400.53,128,0.076892,1.595037
4,C001,A02,2024-10-05,21492,1526,2365.03,134,0.071003,1.549823


In [25]:
# this technique is like a groupby for time-series
df_weekly = df.resample('W', on='date').sum()

# you can calculate neighbor-row calculations
df_weekly['spend_wow'] = df_weekly['spend'].pct_change()

df_weekly.head()

,campaign_id,advertiser_id,impressions,clicks,spend,conversions,ctr,cpc,spend_wow
date,,,,,,,,,
2024-10-06,C001C002C003C004C005C006C007C008C009C010C011C0...,A02A03A04A05A06A01A02A03A04A05A06A01A02A03A04A...,3026042,122968,142560.36,8147,5.026222,155.148677,NaN
2024-10-13,C001C002C003C004C005C006C007C008C009C010C011C0...,A02A03A04A05A06A01A02A03A04A05A06A01A02A03A04A...,3510186,140709,166085.14,9461,5.819987,181.906951,0.165016
2024-10-20,C001C002C003C004C005C006C007C008C009C010C011C0...,A02A03A04A05A06A01A02A03A04A05A06A01A02A03A04A...,3431891,137929,161098.84,9258,5.870452,179.651988,-0.030023
2024-10-27,C001C002C003C004C005C006C007C008C009C010C011C0...,A02A03A04A05A06A01A02A03A04A05A06A01A02A03A04A...,3488449,142135,161898.01,9379,5.877452,179.945217,0.004961
2024-11-03,C001C002C003C004C005C006C007C008C009C010C011C0...,A02A03A04A05A06A01A02A03A04A05A06A01A02A03A04A...,3457029,140855,164995.59,9405,5.914314,180.918313,0.019133


### Q4 — Diagnosis: Anomaly Detection
A PM comes to you and says: 'Something looks off — our conversion rate dropped last week.'

Using the data, **identify which specific campaigns or advertisers are driving the drop**. Walk through your approach in comments before writing the code.

In [ ]:
# first i want to capture the clicks and conversions for a given week for a campaign
campaign_week_df = df.groupby("campaign_id").resample("W", on="date").sum()

# capture the conversion rate: conversion / clicks. this will control for the differences in magnitude of clicks and conversions for campaigns
campaign_week_df["conversion_rate"] = campaign_week_df["conversions"]/campaign_week_df["clicks"]


campaign_week_df

campaign_id          advertiser_id  \
campaign_id date                                                              
C001        2024-10-06      C001C001C001C001C001C001     A02A02A02A02A02A02   
            2024-10-13  C001C001C001C001C001C001C001  A02A02A02A02A02A02A02   
            2024-10-20  C001C001C001C001C001C001C001  A02A02A02A02A02A02A02   
            2024-10-27  C001C001C001C001C001C001C001  A02A02A02A02A02A02A02   
            2024-11-03  C001C001C001C001C001C001C001  A02A02A02A02A02A02A02   
...                                              ...                    ...   
C020        2024-11-03  C020C020C020C020C020C020C020  A03A03A03A03A03A03A03   
            2024-11-10  C020C020C020C020C020C020C020  A03A03A03A03A03A03A03   
            2024-11-17  C020C020C020C020C020C020C020  A03A03A03A03A03A03A03   
            2024-11-24  C020C020C020C020C020C020C020  A03A03A03A03A03A03A03   
            2024-12-01      C020C020C020C020C020C020     A03A03A03A03A03A03   

                        impressions  clicks     spend  conversions       ctr  \
campaign_id date                                                               
C001        2024-10-06       124527    9377  14241.68          882  0.451885   
            2024-10-13       141377   10760  17746.46         1064  0.533012   
            2024-10-20       139152   10623  17026.81          930  0.535788   
            2024-10-27       146493   10970  17314.73         1017  0.524187   
            2024-11-03       145027   11238  18584.56         1072  0.541303   
...                             ...     ...       ...          ...       ...   
C020        2024-11-03        91308    6346   3827.09          602  0.486299   
            2024-11-10        93580    6759   4229.70          613  0.505756   
            2024-11-17        94756    6389   3773.79          558  0.471567   
            2024-11-24        87425    5868   3604.92          488  0.471531   
            2024-12-01        77866    5405   3324.20          463  0.415311   

                              cpc  conversion_rate  
campaign_id date                                    
C001        2024-10-06   9.115063         0.094060  
            2024-10-13  11.536343         0.098885  
            2024-10-20  11.216938         0.087546  
            2024-10-27  11.047698         0.092707  
            2024-11-03  11.605860         0.095391  
...                           ...              ...  
C020        2024-11-03   4.238255         0.094863  
            2024-11-10   4.380058         0.090694  
            2024-11-17   4.138620         0.087338  
            2024-11-24   4.300052         0.083163  
            2024-12-01   3.692491         0.085661  

[180 rows x 9 columns]

### Bonus — Product Question Response

The PM asks: 'We are seeing strong CTR but low conversion rates across several campaigns. What hypotheses would you form, and how would you investigate further?'

> Your answer here...